# LLM Model Evaluator

Evaluate one saved `LLM_v2` artifact run on either the validation or test split and compare it against fixed-chain baselines plus per-circuit upper bounds.


In [ ]:
from pathlib import Path
import json
import os
import sys
import textwrap
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "dbs" / "combined_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

LLM_DIR = PROJECT_ROOT / "machine_learning" / "LLM" / "LLM_v2"
ARTIFACT_ROOT = LLM_DIR / "artifacts"
for path in (PROJECT_ROOT, LLM_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from machine_learning.LLM.LLM_v2.data import candidate_rows_for_circuit, load_dataset_bundle, split_frame_for_name
from machine_learning.LLM.LLM_v2.evaluate import score_circuit_candidates
from machine_learning.LLM.LLM_v2.objective import ObjectiveConfig, objective_metadata

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


## Configure Run

Set `EVAL_SPLIT` to the split you want to inspect. Leave `LLM_RUN_DIR = None` to automatically pick the newest saved artifact run for that split, or point it at a specific run directory manually.


In [ ]:
EVAL_SPLIT = "validation"  # "validation" or "test"
LLM_RUN_DIR = None
SAVE_FIGURES = True
OUTPUT_DIR = LLM_DIR / "evaluator_figures"
WRAP_WIDTH = 22

assert EVAL_SPLIT in {"validation", "test"}


In [ ]:
def resolve_latest_run(artifact_root: Path, split_name: str) -> Path:
    candidates = []
    for run_dir in sorted(artifact_root.iterdir()) if artifact_root.exists() else []:
        if not run_dir.is_dir():
            continue
        manifest_path = run_dir / "manifest.json"
        if not manifest_path.exists():
            continue
        try:
            manifest = json.loads(manifest_path.read_text())
        except json.JSONDecodeError:
            continue
        if str(manifest.get("split", "")).strip().lower() == split_name:
            candidates.append(run_dir)
    if not candidates:
        raise FileNotFoundError(
            f"No saved LLM artifact runs were found for split={split_name!r} under {artifact_root}."
        )
    return candidates[-1]


if LLM_RUN_DIR is None:
    LLM_RUN_DIR = resolve_latest_run(ARTIFACT_ROOT, EVAL_SPLIT)
else:
    LLM_RUN_DIR = Path(LLM_RUN_DIR)
    if not LLM_RUN_DIR.is_absolute():
        LLM_RUN_DIR = (PROJECT_ROOT / LLM_RUN_DIR).resolve()

if not LLM_RUN_DIR.exists():
    raise FileNotFoundError(f"LLM artifact directory not found: {LLM_RUN_DIR}")

manifest_path = LLM_RUN_DIR / "manifest.json"
predictions_path = LLM_RUN_DIR / "predictions.csv"
summary_path = LLM_RUN_DIR / "summary.json"

for path in (manifest_path, predictions_path, summary_path):
    if not path.exists():
        raise FileNotFoundError(f"Required artifact file missing: {path}")

manifest = json.loads(manifest_path.read_text())
run_summary = json.loads(summary_path.read_text())
predictions = pd.read_csv(predictions_path)

required_prediction_columns = [
    "circuit_id",
    "selected_chain_id",
    "selected_chain",
    "objective_score",
    "oracle_best_chain_id",
    "oracle_best_score",
    "regret",
    "two_qubit_gate_reduction_pct",
    "depth_reduction_pct",
    "runtime_seconds",
    "runtime_score",
]
missing_prediction_columns = [
    column for column in required_prediction_columns if column not in predictions.columns
]
if missing_prediction_columns:
    raise ValueError(
        f"Predictions file is missing required columns: {missing_prediction_columns}"
    )

predicted_circuit_ids = sorted(predictions["circuit_id"].astype(str).unique().tolist())
if not predicted_circuit_ids:
    raise ValueError("Predictions file did not contain any circuit ids.")

artifact_split = str(manifest.get("split", "")).strip().lower()
if artifact_split != EVAL_SPLIT:
    raise ValueError(
        f"Configured EVAL_SPLIT={EVAL_SPLIT!r} does not match artifact split={artifact_split!r}."
    )

objective_defaults = ObjectiveConfig()
objective_config = ObjectiveConfig(
    reward_type=str(manifest.get("objective", {}).get("reward_type", objective_defaults.reward_type)),
    runtime_weight=float(manifest.get("objective", {}).get("runtime_weight", objective_defaults.runtime_weight)),
    two_qubit_weight=float(manifest.get("objective", {}).get("two_qubit_weight", objective_defaults.two_qubit_weight)),
    depth_weight=float(manifest.get("objective", {}).get("depth_weight", objective_defaults.depth_weight)),
    final_reward_weight=float(manifest.get("objective", {}).get("final_reward_weight", objective_defaults.final_reward_weight)),
)

artifact_info = pd.DataFrame([
    {
        "run_name": manifest.get("run_name"),
        "split": EVAL_SPLIT,
        "model_name": manifest.get("model_name"),
        "prompt_mode": manifest.get("prompt_mode"),
        "feature_subset": manifest.get("feature_subset"),
        "retrieval_k": manifest.get("retrieval_k"),
        "num_circuits_manifest": manifest.get("num_circuits"),
        "num_prediction_rows": len(predictions),
        "num_unique_prediction_circuits": len(predicted_circuit_ids),
        "artifact_dir": str(LLM_RUN_DIR),
    }
])

display(artifact_info)
display(pd.DataFrame([objective_metadata(objective_config)]))
display(pd.DataFrame([run_summary]))


## Prepare Split Scores

Score all candidate chains on the train split and on exactly the circuits present in the saved LLM artifact. This works for both quick trial runs and full split runs.


In [ ]:
bundle = load_dataset_bundle()
train_df = split_frame_for_name(bundle, "train")
full_eval_df = split_frame_for_name(bundle, EVAL_SPLIT)
eval_df = full_eval_df[full_eval_df["original_circuit"].astype(str).isin(predicted_circuit_ids)].copy()

missing_from_split = sorted(set(predicted_circuit_ids) - set(eval_df["original_circuit"].astype(str).unique()))
if missing_from_split:
    raise ValueError(
        f"Some predicted circuits were not found in the {EVAL_SPLIT} split: {missing_from_split[:10]}"
    )


def score_split_candidates(frame):
    scored_frames = []
    for circuit_id in sorted(frame["original_circuit"].unique().tolist()):
        circuit_rows = candidate_rows_for_circuit(frame, circuit_id)
        scored = score_circuit_candidates(circuit_rows, objective_config).copy()
        scored["original_circuit"] = circuit_id
        scored_frames.append(scored)
    if not scored_frames:
        raise ValueError("No circuits were available to score.")
    return pd.concat(scored_frames, ignore_index=True)


train_scored = score_split_candidates(train_df)
eval_scored = score_split_candidates(eval_df)

split_checks = pd.DataFrame([
    {
        "train_circuits": train_df["original_circuit"].nunique(),
        f"full_{EVAL_SPLIT}_circuits": full_eval_df["original_circuit"].nunique(),
        f"predicted_{EVAL_SPLIT}_circuits": eval_df["original_circuit"].nunique(),
        "train_scored_rows": len(train_scored),
        "eval_scored_rows": len(eval_scored),
        "expected_eval_rows": eval_df["original_circuit"].nunique() * 30,
    }
])
display(split_checks)
display(eval_scored.head())


## Align Saved LLM Selections

Join the saved LLM selections back onto the scored evaluation rows so the notebook can compare the LLM method against fixed-chain baselines and oracle upper bounds.


In [ ]:
train_chain_means = (
    train_scored.groupby(["chain_id", "chain_name"], as_index=False)["objective_score"]
    .mean()
    .sort_values(["objective_score", "chain_id"], ascending=[False, True])
)
global_best_average_chain_id = train_chain_means.iloc[0]["chain_id"]
global_best_average_chain_name = train_chain_means.iloc[0]["chain_name"]

selected_rows = eval_scored.merge(
    predictions,
    left_on=["original_circuit", "chain_id"],
    right_on=["circuit_id", "selected_chain_id"],
    how="inner",
    suffixes=("", "_pred"),
)

expected_circuit_count = len(predicted_circuit_ids)
if len(selected_rows) != expected_circuit_count:
    missing = sorted(set(predicted_circuit_ids) - set(selected_rows["original_circuit"].astype(str).unique()))
    raise ValueError(
        f"Expected {expected_circuit_count} selected rows after merge, found {len(selected_rows)}. "
        f"Missing circuits: {missing[:10]}"
    )

selected_rows = selected_rows.copy()
selected_rows["method"] = f"LLM ({manifest.get('model_name', 'unknown model')})"
selected_rows["selected_chain"] = selected_rows["chain_name"]
selected_rows["confidence"] = pd.to_numeric(selected_rows.get("confidence"), errors="coerce")
selected_rows["used_fallback"] = (
    selected_rows.get("used_fallback", False).astype(str).str.lower().eq("true")
)
selected_rows = selected_rows.sort_values("original_circuit").reset_index(drop=True)

selection_checks = pd.DataFrame([
    {
        "selected_circuits": len(selected_rows),
        "expected_circuits": expected_circuit_count,
        "mean_objective_score_from_predictions": pd.to_numeric(predictions["objective_score"], errors="coerce").mean(),
        "mean_objective_score_from_join": selected_rows["objective_score"].mean(),
        "mean_regret": pd.to_numeric(selected_rows["regret"], errors="coerce").mean(),
        "fallback_rate": float(selected_rows["used_fallback"].mean()),
        "mean_confidence": float(selected_rows["confidence"].dropna().mean()) if selected_rows["confidence"].notna().any() else np.nan,
        "global_best_average_chain_id": global_best_average_chain_id,
        "global_best_average_chain_name": global_best_average_chain_name,
    }
])
display(selection_checks)
display(selected_rows.head())


In [ ]:
def fixed_chain_rows(frame, chain_name, method_name):
    rows = frame[frame["chain_name"] == chain_name].copy()
    rows["method"] = method_name
    rows["selected_chain"] = rows["chain_name"]
    rows["regret"] = np.nan
    rows["used_fallback"] = False
    rows["confidence"] = np.nan
    return rows


def best_possible_objective_rows(frame, method_name):
    rows = frame.loc[frame.groupby("original_circuit")["objective_score"].idxmax()].copy()
    rows["method"] = method_name
    rows["selected_chain"] = rows["chain_name"]
    rows["regret"] = 0.0
    rows["used_fallback"] = False
    rows["confidence"] = np.nan
    return rows


def best_possible_depth_rows(frame, method_name):
    rows = frame.loc[frame.groupby("original_circuit")["total_depth_reduction_pct"].idxmax()].copy()
    rows["method"] = method_name
    rows["selected_chain"] = rows["chain_name"]
    rows["regret"] = np.nan
    rows["used_fallback"] = False
    rows["confidence"] = np.nan
    return rows


def best_possible_twoq_rows(frame, method_name):
    rows = frame.loc[frame.groupby("original_circuit")["total_two_qubit_gate_reduction_pct"].idxmax()].copy()
    rows["method"] = method_name
    rows["selected_chain"] = rows["chain_name"]
    rows["regret"] = np.nan
    rows["used_fallback"] = False
    rows["confidence"] = np.nan
    return rows


def summarize_selected(rows, method_name):
    return {
        "method": method_name,
        "circuits": len(rows),
        "avg_objective_score": float(pd.to_numeric(rows["objective_score"], errors="coerce").mean()),
        "avg_regret": float(pd.to_numeric(rows["regret"], errors="coerce").mean()) if rows["regret"].notna().any() else np.nan,
        "avg_two_qubit_gate_reduction_pct": float(pd.to_numeric(rows["total_two_qubit_gate_reduction_pct"], errors="coerce").mean()),
        "avg_depth_reduction_pct": float(pd.to_numeric(rows["total_depth_reduction_pct"], errors="coerce").mean()),
        "avg_runtime_seconds": float(pd.to_numeric(rows["total_runtime_seconds"], errors="coerce").mean()),
        "avg_runtime_score": float(pd.to_numeric(rows["total_runtime_score"], errors="coerce").mean()),
        "fallback_rate": float(pd.to_numeric(rows["used_fallback"], errors="coerce").fillna(0).mean()) if "used_fallback" in rows else np.nan,
        "mean_confidence": float(pd.to_numeric(rows["confidence"], errors="coerce").mean()) if "confidence" in rows else np.nan,
    }


single_chains = sorted(name for name in train_scored[train_scored["length"] == 1]["chain_name"].unique())
double_chains = sorted(name for name in train_scored[train_scored["length"] == 2]["chain_name"].unique())

best_fixed_reward_chain = train_scored.groupby("chain_name")["objective_score"].mean().sort_values(ascending=False).index[0]
best_single_optimizer_for_depth_reduction = (
    train_scored[train_scored["chain_name"].isin(single_chains)]
    .groupby("chain_name")["total_depth_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)
best_single_optimizer_for_two_qubit_reduction = (
    train_scored[train_scored["chain_name"].isin(single_chains)]
    .groupby("chain_name")["total_two_qubit_gate_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)
best_chain_for_depth_reduction = (
    train_scored[train_scored["chain_name"].isin(double_chains)]
    .groupby("chain_name")["total_depth_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)
best_chain_for_two_qubit_reduction = (
    train_scored[train_scored["chain_name"].isin(double_chains)]
    .groupby("chain_name")["total_two_qubit_gate_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

llm_method_name = str(selected_rows["method"].iloc[0])
best_reward_chain_label = f"Best fixed reward chain ({best_fixed_reward_chain})"
best_depth_chain_label = f"Best depth reduction chain ({best_chain_for_depth_reduction})"
best_twoq_chain_label = f"Best 2Q gate reduction chain ({best_chain_for_two_qubit_reduction})"
best_depth_single_label = f"Best depth reduction single ({best_single_optimizer_for_depth_reduction})"
best_twoq_single_label = f"Best 2Q gate reduction single ({best_single_optimizer_for_two_qubit_reduction})"
best_possible_objective_label = "Best possible objective"
best_possible_depth_label = "Best possible depth reduction"
best_possible_twoq_label = "Best possible two-qubit gate reduction"

baseline_selected_frames = {
    best_reward_chain_label: fixed_chain_rows(eval_scored, best_fixed_reward_chain, best_reward_chain_label),
    best_depth_chain_label: fixed_chain_rows(eval_scored, best_chain_for_depth_reduction, best_depth_chain_label),
    best_twoq_chain_label: fixed_chain_rows(eval_scored, best_chain_for_two_qubit_reduction, best_twoq_chain_label),
    best_depth_single_label: fixed_chain_rows(eval_scored, best_single_optimizer_for_depth_reduction, best_depth_single_label),
    best_twoq_single_label: fixed_chain_rows(eval_scored, best_single_optimizer_for_two_qubit_reduction, best_twoq_single_label),
    best_possible_objective_label: best_possible_objective_rows(eval_scored, best_possible_objective_label),
    best_possible_depth_label: best_possible_depth_rows(eval_scored, best_possible_depth_label),
    best_possible_twoq_label: best_possible_twoq_rows(eval_scored, best_possible_twoq_label),
}

comparison = pd.DataFrame(
    [summarize_selected(selected_rows, llm_method_name)]
    + [summarize_selected(frame, method) for method, frame in baseline_selected_frames.items()]
)
comparison = comparison.sort_values("avg_objective_score", ascending=False).reset_index(drop=True)
display(comparison)


## Visual Comparisons

These plots mirror the RL v3 evaluator style: chain-count heatmap, average metric bars, and circuit-by-circuit comparison against the oracle objective.


In [ ]:
def prettify_label(name, wrap_width=22):
    name = str(name).replace("__", " --> ").replace("_", " ")
    replacements = {
        "qiskit standard": "Qiskit",
        "qiskit ai": "Qiskit AI",
        "wisq rules": "WISQ Rules",
        "wisq bqskit": "WISQ BQSKit",
        "tket": "TKET",
        "best fixed reward chain": "Best Fixed Reward Chain",
        "best depth reduction chain": "Best Depth Reduction Chain",
        "best 2q gate reduction chain": "Best 2Q Gate Reduction Chain",
        "best depth reduction single": "Best Depth Reduction Single",
        "best 2q gate reduction single": "Best 2Q Gate Reduction Single",
        "best possible objective": "Best Possible Objective",
        "best possible depth reduction": "Best Possible Depth Reduction",
        "best possible two-qubit gate reduction": "Best Possible Two-Qubit Gate Reduction",
        "llm": "LLM",
    }
    lowered = name.lower()
    for old, new in replacements.items():
        lowered = lowered.replace(old, new)
    pretty = lowered
    if wrap_width is None:
        return pretty
    return textwrap.fill(pretty, width=wrap_width)


def save_figure(fig, stem):
    if not SAVE_FIGURES:
        return
    target_dir = OUTPUT_DIR / str(manifest.get("run_name", LLM_RUN_DIR.name))
    target_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(target_dir / f"{stem}.png", bbox_inches="tight")
    fig.savefig(target_dir / f"{stem}.pdf", bbox_inches="tight")


def horizontal_bar(data, x, xlabel, title, color="#4C78A8", fmt="{:.3f}", filename=None):
    plot_data = data.sort_values(x).copy()
    plot_data["method_pretty"] = plot_data["method"].apply(lambda name: prettify_label(name, wrap_width=WRAP_WIDTH))

    fig_height = max(3.0, 0.55 * len(plot_data))
    fig, ax = plt.subplots(figsize=(5.8, fig_height))
    ax.grid(False)
    bars = ax.barh(plot_data["method_pretty"], plot_data[x], color=color)
    ax.set_title(title, fontsize=11, pad=10)
    ax.set_xlabel(xlabel, fontsize=10, labelpad=8)

    max_value = plot_data[x].max()
    min_value = plot_data[x].min()
    value_range = max_value - min_value
    if value_range == 0:
        value_range = abs(max_value) if max_value != 0 else 1
    padding = 0.03 * value_range

    for bar, value in zip(bars, plot_data[x]):
        ax.text(bar.get_width() + padding, bar.get_y() + bar.get_height() / 2, fmt.format(value), va="center", fontsize=8)

    ax.set_xlim(min(0, min_value) - 0.05 * value_range, max_value + 0.22 * value_range)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    if filename:
        save_figure(fig, filename)
    plt.show()


plot_frames = {llm_method_name: selected_rows, **baseline_selected_frames}
selected_chain_counts = pd.concat(
    [frame[["method", "selected_chain"]] for frame in plot_frames.values()],
    ignore_index=True,
).groupby(["method", "selected_chain"]).size().reset_index(name="circuits")

selected_chain_grid = selected_chain_counts.pivot_table(
    index="method",
    columns="selected_chain",
    values="circuits",
    fill_value=0,
    aggfunc="sum",
)

method_order = [
    llm_method_name,
    best_reward_chain_label,
    best_depth_chain_label,
    best_twoq_chain_label,
    best_depth_single_label,
    best_twoq_single_label,
    best_possible_objective_label,
    best_possible_depth_label,
    best_possible_twoq_label,
]
selected_chain_grid = selected_chain_grid.loc[
    [method for method in method_order if method in selected_chain_grid.index],
    selected_chain_grid.sum(axis=0).sort_values(ascending=False).index,
]

selected_chain_grid_pretty = selected_chain_grid.copy()
selected_chain_grid_pretty.index = [prettify_label(name, wrap_width=24) for name in selected_chain_grid_pretty.index]
selected_chain_grid_pretty.columns = [prettify_label(name, wrap_width=18) for name in selected_chain_grid_pretty.columns]

display(selected_chain_grid_pretty)

fig, ax = plt.subplots(figsize=(max(12, 0.6 * selected_chain_grid_pretty.shape[1]), 5.0))
sns.heatmap(
    selected_chain_grid_pretty,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": f"Number of evaluated {EVAL_SPLIT} circuits"},
    ax=ax,
)
ax.set_xlabel("Selected optimizer chain")
ax.set_ylabel("Method")
ax.set_title("Selected Optimizer Chain Counts by Method")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
save_figure(fig, "selected_chain_counts_heatmap")
plt.show()

horizontal_bar(comparison, "avg_objective_score", "Average objective score", f"Average objective score on evaluated {EVAL_SPLIT.title()} circuits", filename="avg_objective_score")
horizontal_bar(comparison, "avg_two_qubit_gate_reduction_pct", "Average 2Q reduction (%)", f"Average 2Q reduction on evaluated {EVAL_SPLIT.title()} circuits", filename="avg_two_qubit_reduction")
horizontal_bar(comparison, "avg_depth_reduction_pct", "Average depth reduction (%)", f"Average depth reduction on evaluated {EVAL_SPLIT.title()} circuits", filename="avg_depth_reduction")
horizontal_bar(comparison, "avg_runtime_seconds", "Average runtime (seconds)", f"Average runtime on evaluated {EVAL_SPLIT.title()} circuits", color="#72B7B2", filename="avg_runtime_seconds")

regret_plot = comparison.dropna(subset=["avg_regret"]).copy()
if not regret_plot.empty:
    horizontal_bar(regret_plot, "avg_regret", "Average regret", f"Average regret on evaluated {EVAL_SPLIT.title()} circuits", color="#E45756", filename="avg_regret")


In [ ]:
oracle_compare = (
    selected_rows.set_index("original_circuit")[["selected_chain", "objective_score", "regret"]]
    .rename(columns={
        "selected_chain": "llm_selected_chain",
        "objective_score": "llm_objective_score",
        "regret": "llm_regret",
    })
    .join(
        baseline_selected_frames[best_possible_objective_label]
        .set_index("original_circuit")[["chain_name", "objective_score"]]
        .rename(columns={
            "chain_name": "oracle_chain",
            "objective_score": "oracle_objective_score",
        }),
        how="inner",
    )
)
oracle_compare["llm_minus_oracle"] = oracle_compare["llm_objective_score"] - oracle_compare["oracle_objective_score"]
oracle_compare = oracle_compare.sort_values("llm_minus_oracle")

display(oracle_compare)

fig, ax = plt.subplots(figsize=(14, 4.8))
positions = np.arange(len(oracle_compare.index))
ax.bar(
    positions,
    oracle_compare["llm_minus_oracle"],
    color=np.where(oracle_compare["llm_minus_oracle"] >= 0, "#4C78A8", "#E45756"),
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(positions)
ax.set_xticklabels(oracle_compare.index, rotation=90)
ax.set_ylabel("LLM objective - oracle objective")
ax.set_title(f"LLM Objective Difference vs Oracle on Each Evaluated {EVAL_SPLIT.title()} Circuit")
plt.tight_layout()
save_figure(fig, "llm_vs_oracle_per_circuit")
plt.show()
